# 01 深度学习推理基础：只讲 AI Infra 必须懂的部分

本章目标是让你能读懂后面所有推理服务内容。我们不讲训练细节，不讲反向传播，不讲优化器，只讲推理时数据如何流动：token 变成向量，向量经过 attention 和 MLP，最后变成下一个 token 的概率分布。

如果你是应用开发者，可以把模型理解成一个巨大的函数：输入 token 序列，输出下一个 token 的分数。AI Infra 要优化的，就是这个函数在高并发、长上下文、有限显存下的执行效率和稳定性。


## 1. 从 token 到 tensor

自然语言不能直接进入神经网络。第一步是 tokenizer，把文本切成 token，并映射成整数 id。

```mermaid
flowchart LR
    Text["文本<br/>AI Infra 很重要"] --> Tok["分词器 / Tokenizer"]
    Tok --> Ids["Token 编号 / token ids<br/>[101, 42, 998, ...]"]
    Ids --> Emb["嵌入表查找 / Embedding lookup"]
    Emb --> Tensor["向量序列<br/>shape = seq_len x hidden_dim"]
```

你需要掌握三个 shape：

- `seq_len`：序列长度，也就是 token 数。长 prompt 会让这个值变大。
- `hidden_dim`：每个 token 的向量维度。模型越大通常 hidden_dim 越大。
- `batch_size`：一次并行处理多少条序列。推理服务里的 batching 会动态变化。

Embedding table 本质上是一个矩阵，shape 约为 `vocab_size x hidden_dim`。token id 是行号，查表得到该 token 的向量。


In [ ]:
import numpy as np

vocab = {'<pad>': 0, 'AI': 1, 'Infra': 2, '很': 3, '重要': 4}
ids = np.array([vocab['AI'], vocab['Infra'], vocab['很'], vocab['重要']])

rng = np.random.default_rng(0)
vocab_size = len(vocab)
hidden_dim = 6
embedding_table = rng.normal(size=(vocab_size, hidden_dim))

x = embedding_table[ids]
print('token ids:', ids)
print('embedding_table shape:', embedding_table.shape)
print('x shape:', x.shape, '= seq_len x hidden_dim')
print(np.round(x, 3))


## 2. 矩阵乘法为什么是核心

Transformer 里大量计算都是矩阵乘法：embedding 后的向量会乘以不同权重矩阵，得到 Query、Key、Value、MLP 中间表示和 logits。GPU 适合 LLM 推理，正是因为它擅长大规模并行矩阵计算。

一个线性层可以写成：

$$
Y = XW + b
$$

其中 `X` 是输入 tensor，`W` 是模型权重，`b` 是偏置。推理时，权重已经训练好，不会更新；系统要做的是快速把很多请求的 `X` 送进去，得到输出。

这也解释了为什么模型参数量会影响基础显存：参数本质上就是大量矩阵的元素。


In [ ]:
# 一个线性层的 shape 变化。
seq_len, hidden_dim, out_dim = 4, 6, 10
x = rng.normal(size=(seq_len, hidden_dim))
w = rng.normal(size=(hidden_dim, out_dim))
b = rng.normal(size=(out_dim,))
y = x @ w + b
print('X:', x.shape)
print('W:', w.shape)
print('Y:', y.shape)


## 3. Attention：模型如何“看上下文”

Self-attention 的作用是：对每个 token，计算它应该关注上下文中哪些 token。它会把输入向量变成三组向量：

- **Query (Q)**：当前 token 想找什么信息。
- **Key (K)**：每个历史 token 提供什么索引。
- **Value (V)**：真正被聚合的信息。

核心公式：

$$
\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

读公式不必害怕：`QK^T` 是“当前 token 和所有 token 的匹配分数”，`softmax` 把分数变成权重，最后乘以 `V` 得到加权后的上下文信息。

Infra 视角最重要的是复杂度：prompt 越长，attention 要比较的 token 对越多。prefill 阶段处理完整 prompt，计算量近似随长度平方增长；decode 阶段每次新增一个 token，但要读取历史 KV cache。


In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

seq_len, hidden = 5, 8
x = rng.normal(size=(seq_len, hidden))
wq = rng.normal(size=(hidden, hidden)) / hidden**0.5
wk = rng.normal(size=(hidden, hidden)) / hidden**0.5
wv = rng.normal(size=(hidden, hidden)) / hidden**0.5

q, k, v = x @ wq, x @ wk, x @ wv
scores = q @ k.T / np.sqrt(hidden)
weights = softmax(scores)
out = weights @ v

print('q shape:', q.shape)
print('k shape:', k.shape)
print('scores shape:', scores.shape, '= seq_len x seq_len')
print('attention output shape:', out.shape)
print('row sums:', weights.sum(axis=1).round(4))


## 4. MLP、残差与 Transformer block

一个 Transformer block 通常包含 attention 和 MLP。Attention 负责混合上下文信息，MLP 负责对每个位置做非线性变换。实际模型还有 layer norm、residual connection、RoPE 等结构，但先抓住主线即可。

```mermaid
flowchart LR
    X["输入向量序列"] --> LN1["层归一化 / LayerNorm"]
    LN1 --> Attn["自注意力 / Self-Attention"]
    Attn --> Add1["残差相加 / Residual Add"]
    X --> Add1
    Add1 --> LN2["层归一化 / LayerNorm"]
    LN2 --> MLP["前馈网络 / MLP / FFN"]
    MLP --> Add2["残差相加 / Residual Add"]
    Add1 --> Add2
    Add2 --> Y["下一层输入"]
```

残差连接的直觉是：每层不是完全重写表示，而是在原表示基础上增加一部分更新。对 Infra 来说，你不需要推导这些结构，但需要知道它们都消耗计算和显存，层数越多、hidden_dim 越大，模型越重。


## 5. Logits 与 sampling

模型最后输出 logits，也就是每个候选 token 的分数。把 logits 经过 softmax 后得到概率，再根据策略选择下一个 token。

常见采样参数：

| 参数 | 作用 | 适用场景 |
|---|---|---|
| temperature | 控制分布尖锐/发散 | 低温适合稳定任务，高温适合创意 |
| top-p | 只保留累计概率达到 p 的候选 | 常用开放式生成 |
| top-k | 只保留分数最高的 k 个候选 | 限制候选范围 |
| max_tokens | 最多生成多少 token | 控制成本和延迟 |
| stop | 遇到指定字符串停止 | 工具调用/结构化输出 |

评测和复现时，采样参数必须固定。否则你无法判断结果变化来自模型、prompt、数据，还是随机采样。


In [ ]:
import math

vocab = ['安全', '快速', '昂贵', '稳定', '复杂']
logits = np.array([3.0, 2.0, 0.5, 2.5, 1.0])

for temperature in [0.3, 0.8, 1.5]:
    probs = softmax(logits / temperature)
    print('temperature =', temperature)
    for token, p in zip(vocab, probs):
        print(f'  {token:4s} {p:.3f}')


## 6. 训练和推理的区别

```mermaid
flowchart LR
    Data["训练数据"] --> Train["训练<br/>反向传播 + 更新权重"]
    Train --> Weights["模型权重"]
    Prompt["用户 prompt"] --> Infer["推理<br/>只做前向计算"]
    Weights --> Infer
    Infer --> Output["输出 tokens"]
```

训练需要保存大量中间状态用于反向传播，还要更新权重；推理只做前向计算，不更新权重。因此推理的主要工程问题变成：权重如何装进显存，KV cache 如何管理，很多用户如何共享 GPU，输出如何低延迟流式返回。

本教程后面几章都会围绕推理展开。


## 7. 常见误区

- **误区 1：模型推理只和模型大小有关。** 实际上上下文长度、并发、输出长度、sampling、batching 都会影响延迟和显存。
- **误区 2：token 和字数差不多。** 中文、英文、代码、标点的 tokenization 差异很大，容量规划必须按 token 看。
- **误区 3：temperature 只是“创意程度”。** 它还影响评测稳定性和结构化输出可靠性。
- **误区 4：会调 API 就等于懂模型。** AI Infra 至少要知道模型执行的资源结构，否则很难解释性能问题。


## 8. Shape 是理解模型推理的调试语言

初学深度学习时，最容易被复杂术语吓住。实际工程里，先把 shape 看懂，很多问题就清晰了。假设 batch size 为 `B`，序列长度为 `S`，隐藏维度为 `H`，词表大小为 `V`：

| 阶段 | Tensor shape | 含义 |
|---|---|---|
| token ids | `B x S` | 每个请求的 token id 序列 |
| embeddings | `B x S x H` | 每个 token 的隐藏向量 |
| Q/K/V | `B x S x H` 或按 head 展开 | attention 的查询、键、值 |
| attention scores | `B x heads x S x S` | 每个 token 对每个 token 的注意力分数 |
| hidden states | `B x S x H` | 每层 Transformer 的输出 |
| logits | `B x S x V` | 每个位置对所有词表 token 的分数 |

在 decode 阶段，新的输入通常只有一个 token，所以 `S` 对当前 step 来说是 1，但模型会读取历史 KV cache。这个差异解释了为什么 prefill 和 decode 性能特征不同。


In [ ]:
# 观察一个极简 batch 的 shape 流动。
B, S, H, V = 2, 4, 8, 20
rng = np.random.default_rng(123)
token_ids = rng.integers(0, V, size=(B, S))
embedding = rng.normal(size=(V, H))
x = embedding[token_ids]
w_logits = rng.normal(size=(H, V)) / H**0.5
logits = x @ w_logits

print('token_ids:', token_ids.shape)
print('x embeddings:', x.shape)
print('logits:', logits.shape)
print('next-token logits for last position:', logits[:, -1, :].shape)


## 9. 为什么上下文窗口不是“免费记忆”

上下文窗口越大，模型可以读取的历史越长，但代价也越高。长上下文至少带来三类成本：

1. **Prefill 计算成本**：长 prompt 进入模型时，attention 要处理更多 token 关系。
2. **KV cache 显存成本**：每个历史 token 的 K/V 都要保存，活跃请求越多成本越高。
3. **质量成本**：模型未必能有效利用所有上下文，噪声文档可能干扰回答。

所以生产 RAG 不应该默认“能塞多少塞多少”。更好的路径是：提高检索质量、rerank、压缩上下文、保留证据和引用，而不是盲目扩大 context window。


## 10. 面试/工作表达

如果别人问你“你不是算法背景，怎么理解 Transformer 推理？”，可以这样答：

> 我不会从训练推导开始讲，而是从推理数据流讲：文本先被 tokenizer 变成 token ids，再通过 embedding 变成 `seq_len x hidden_dim` 的向量。Transformer block 主要由 attention 和 MLP 组成，attention 用 Q/K/V 计算上下文相关性，最后输出 logits，再通过 sampling 得到下一个 token。Infra 角度最重要的是 shape、上下文长度、KV cache 和采样参数，因为它们直接影响显存、延迟、吞吐和复现。


## 11. 把 Transformer 推理串成一条完整链路

现在把前面的基础串起来。用户输入一句话，tokenizer 先把文本切成 token ids。Embedding 层把 ids 查表成向量。每一层 Transformer block 会先用 attention 混合上下文，再用 MLP 做非线性变换。最后一层输出 hidden states，乘以词表投影矩阵得到 logits。服务端再根据 temperature、top-p、top-k、stop 等参数选择下一个 token。这个 token 会被追加到上下文里，进入下一轮 decode。

如果是 prefill，模型一次处理 prompt 中所有 token，因此 attention scores 的形状里会出现 `S x S`。如果是 decode，当前输入通常只有新 token，但它会读取之前保存的 KV cache。因此你可以这样记：prefill 的成本来自“把整段 prompt 看一遍”，decode 的成本来自“每生成一步都要读历史”。

这条链路里，AI Infra 关心的不只是模型是否聪明，而是每一步的资源含义。Tokenizer 可能成为 CPU 瓶颈；embedding 和矩阵乘法消耗显存带宽和算力；attention 受上下文长度影响；logits 和 sampling 影响输出稳定性；KV cache 影响并发和长上下文成本。后续 serving 框架的所有优化，基本都围绕这些点展开。


## 12. 术语表

- **Token**：模型处理文本的基本单位，不等于字或词。
- **Tokenizer**：文本和 token ids 之间的转换器。
- **Embedding**：token id 到向量的查表结果。
- **Hidden state**：模型中间层的 token 表示。
- **Attention head**：attention 的一个子空间，多头让模型从不同角度看上下文。
- **Logits**：softmax 前的词表分数。
- **Sampling**：从 logits 到下一个 token 的选择过程。
- **Context window**：模型一次能看到的最大 token 数。
- **Parameter**：模型权重，推理时固定不更新。
- **KV cache**：decode 阶段保存历史 Key/Value 的缓存。

如果这些词还不熟，不要急着看 vLLM 参数。先能用自己的话解释它们，后面的推理系统会轻松很多。


## 13. 自测题

1. 为什么 logits 不是最终答案，而只是候选 token 分数？
2. `seq_len x seq_len` 的 attention scores 为什么会让长 prompt 成本上升？
3. 训练和推理都使用模型权重，但为什么资源需求不同？
4. temperature 调高会带来什么产品风险？
5. 如果一个 RAG prompt 很长，应该先怀疑模型能力，还是先看上下文和检索策略？


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
